<a href="https://colab.research.google.com/github/divyanshuraj25/Day_17_Advanced_Metadata_RAG/blob/main/Day_17_Metadata_Filtered_RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q sentence-transformers faiss-cpu pandas numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 79.2 MB/s eta 0:00:00


In [2]:
# Day 17 - Metadata Documents

documents = [
    {
        "text": """
        Python is a high-level programming language used for web development,
        automation, data analysis, artificial intelligence and machine learning.
        Python has simple and readable syntax.
        """,
        "metadata": {
            "source": "python_guide.pdf",
            "category": "programming",
            "date": "2026-09-01",
            "document_type": "technical"
        }
    },

    {
        "text": """
        Machine learning is a branch of artificial intelligence.
        It allows computers to learn patterns from data and make predictions
        without being explicitly programmed for every task.
        """,
        "metadata": {
            "source": "ai_handbook.pdf",
            "category": "AI",
            "date": "2026-08-15",
            "document_type": "technical"
        }
    },

    {
        "text": """
        Deep learning uses artificial neural networks with multiple layers.
        It is widely used in computer vision, natural language processing,
        speech recognition and other artificial intelligence applications.
        """,
        "metadata": {
            "source": "deep_learning.pdf",
            "category": "AI",
            "date": "2025-06-20",
            "document_type": "technical"
        }
    },

    {
        "text": """
        Artificial intelligence is being used in marketing to understand
        customer behavior, personalize recommendations and analyze market trends.
        Data can help companies make better marketing decisions.
        """,
        "metadata": {
            "source": "marketing_report.pdf",
            "category": "marketing",
            "date": "2024-03-10",
            "document_type": "report"
        }
    },

    {
        "text": """
        Cloud computing provides computing resources such as servers,
        storage and databases over the internet. It allows organizations
        to scale their applications according to their requirements.
        """,
        "metadata": {
            "source": "cloud_computing.pdf",
            "category": "cloud",
            "date": "2026-01-12",
            "document_type": "technical"
        }
    }
]

print("Total documents:", len(documents))

for doc in documents:
    print("\nSource:", doc["metadata"]["source"])
    print("Category:", doc["metadata"]["category"])
    print("Date:", doc["metadata"]["date"])
    print("Type:", doc["metadata"]["document_type"])

Total documents: 5

Source: python_guide.pdf
Category: programming
Date: 2026-09-01
Type: technical

Source: ai_handbook.pdf
Category: AI
Date: 2026-08-15
Type: technical

Source: deep_learning.pdf
Category: AI
Date: 2025-06-20
Type: technical

Source: marketing_report.pdf
Category: marketing
Date: 2024-03-10
Type: report

Source: cloud_computing.pdf
Category: cloud
Date: 2026-01-12
Type: technical


In [3]:
# Day 17 - Step 3
# Chunk documents while preserving metadata

def create_chunks(documents, chunk_size=80, overlap=20):
    chunks = []

    for doc in documents:
        text = doc["text"].strip()
        metadata = doc["metadata"]

        start = 0

        while start < len(text):
            end = start + chunk_size
            chunk_text = text[start:end]

            chunks.append({
                "text": chunk_text,
                "metadata": metadata.copy()
            })

            start += chunk_size - overlap

    return chunks


# Create chunks
chunks = create_chunks(documents)

print("Total chunks:", len(chunks))

# Display first 3 chunks
for i, chunk in enumerate(chunks[:3]):
    print("\n-----------------------------")
    print("Chunk:", i)
    print("Text:", chunk["text"])
    print("Metadata:", chunk["metadata"])

Total chunks: 20

-----------------------------
Chunk: 0
Text: Python is a high-level programming language used for web development,
        au
Metadata: {'source': 'python_guide.pdf', 'category': 'programming', 'date': '2026-09-01', 'document_type': 'technical'}

-----------------------------
Chunk: 1
Text: elopment,
        automation, data analysis, artificial intelligence and machine
Metadata: {'source': 'python_guide.pdf', 'category': 'programming', 'date': '2026-09-01', 'document_type': 'technical'}

-----------------------------
Chunk: 2
Text: lligence and machine learning.
        Python has simple and readable syntax.
Metadata: {'source': 'python_guide.pdf', 'category': 'programming', 'date': '2026-09-01', 'document_type': 'technical'}


In [4]:
# Day 17 - Step 4
# Create embeddings for all chunks

from sentence_transformers import SentenceTransformer
import numpy as np

# Load embedding model
model = SentenceTransformer("all-MiniLM-L6-v2")

# Extract chunk text
texts = [chunk["text"] for chunk in chunks]

# Generate embeddings
embeddings = model.encode(
    texts,
    convert_to_numpy=True,
    show_progress_bar=True
)

print("Embedding shape:", embeddings.shape)
print("Number of chunks:", len(chunks))
print("Embedding dimension:", embeddings.shape[1])

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Embedding shape: (20, 384)
Number of chunks: 20
Embedding dimension: 384


In [5]:
# Day 17 - Step 5
# Create FAISS vector index

import faiss

# Get embedding dimension
dimension = embeddings.shape[1]

# Create FAISS index
index = faiss.IndexFlatL2(dimension)

# Convert embeddings to float32
embeddings_float32 = embeddings.astype("float32")

# Add embeddings to FAISS
index.add(embeddings_float32)

print("FAISS index created successfully!")
print("Total vectors stored:", index.ntotal)
print("Vector dimension:", dimension)

FAISS index created successfully!
Total vectors stored: 20
Vector dimension: 384


In [6]:
# Day 17 - Step 6
# Normal / Unfiltered Retrieval

def retrieve(query, top_k=3):
    # Convert query into embedding
    query_embedding = model.encode(
        [query],
        convert_to_numpy=True
    ).astype("float32")

    # Search FAISS
    distances, indices = index.search(query_embedding, top_k)

    results = []

    for distance, idx in zip(distances[0], indices[0]):
        result = {
            "text": chunks[idx]["text"],
            "metadata": chunks[idx]["metadata"],
            "distance": float(distance)
        }

        results.append(result)

    return results


# Test retrieval
query = "What is machine learning?"

results = retrieve(query, top_k=3)

print("Query:", query)
print("\nRetrieved Results:")

for i, result in enumerate(results, 1):
    print("\n-----------------------------")
    print("Result:", i)
    print("Distance:", result["distance"])
    print("Text:", result["text"])
    print("Metadata:", result["metadata"])

Query: What is machine learning?

Retrieved Results:

-----------------------------
Result: 1
Distance: 0.4851286709308624
Text: Machine learning is a branch of artificial intelligence.
        It allows compu
Metadata: {'source': 'ai_handbook.pdf', 'category': 'AI', 'date': '2026-08-15', 'document_type': 'technical'}

-----------------------------
Result: 2
Distance: 0.7962075471878052
Text:      It allows computers to learn patterns from data and make predictions
      
Metadata: {'source': 'ai_handbook.pdf', 'category': 'AI', 'date': '2026-08-15', 'document_type': 'technical'}

-----------------------------
Result: 3
Distance: 1.0653676986694336
Text: Deep learning uses artificial neural networks with multiple layers.
        It i
Metadata: {'source': 'deep_learning.pdf', 'category': 'AI', 'date': '2025-06-20', 'document_type': 'technical'}


In [8]:
# Day 17 - Step 7
# Metadata Filtered Retrieval

def filtered_retrieve(query, filters=None, top_k=3):

    # If no filters are provided, use normal retrieval
    if not filters:
        return retrieve(query, top_k=top_k)

    # Find chunks that match the metadata filters
    matching_indices = []

    for i, chunk in enumerate(chunks):
        metadata = chunk["metadata"]

        match = True

        for key, value in filters.items():

            # Date filter
            if key == "cutoff_date":
                if metadata["date"] < value:
                    match = False

            # Exact metadata filter
            elif key in metadata:
                if metadata[key] != value:
                    match = False

        if match:
            matching_indices.append(i)

    # If no documents match the filters
    if len(matching_indices) == 0:
        return []

    # Create embeddings only for matching chunks
    filtered_embeddings = embeddings[matching_indices].astype("float32")

    # Create temporary FAISS index
    dimension = filtered_embeddings.shape[1]
    filtered_index = faiss.IndexFlatL2(dimension)

    filtered_index.add(filtered_embeddings)

    # Embed query
    query_embedding = model.encode(
        [query],
        convert_to_numpy=True
    ).astype("float32")

    # Search only inside filtered documents
    k = min(top_k, len(matching_indices))

    distances, local_indices = filtered_index.search(
        query_embedding,
        k
    )

    results = []

    for distance, local_idx in zip(
        distances[0],
        local_indices[0]
    ):
        original_idx = matching_indices[local_idx]

        results.append({
            "text": chunks[original_idx]["text"],
            "metadata": chunks[original_idx]["metadata"],
            "distance": float(distance)
        })

    return results


print("filtered_retrieve() function created successfully!")

filtered_retrieve() function created successfully!


In [9]:
# Test metadata filtering

query = "What is artificial intelligence?"

filters = {
    "category": "AI"
}

results = filtered_retrieve(
    query,
    filters,
    top_k=3
)

print("Query:", query)
print("Filters:", filters)

print("\nFiltered Results:")

for i, result in enumerate(results, 1):
    print("\n-----------------------------")
    print("Result:", i)
    print("Distance:", result["distance"])
    print("Text:", result["text"])
    print("Metadata:", result["metadata"])

Query: What is artificial intelligence?
Filters: {'category': 'AI'}

Filtered Results:

-----------------------------
Result: 1
Distance: 0.8199464082717896
Text: Machine learning is a branch of artificial intelligence.
        It allows compu
Metadata: {'source': 'ai_handbook.pdf', 'category': 'AI', 'date': '2026-08-15', 'document_type': 'technical'}

-----------------------------
Result: 2
Distance: 1.043189525604248
Text: artificial intelligence applications.
Metadata: {'source': 'deep_learning.pdf', 'category': 'AI', 'date': '2025-06-20', 'document_type': 'technical'}

-----------------------------
Result: 3
Distance: 1.0669026374816895
Text:      It allows computers to learn patterns from data and make predictions
      
Metadata: {'source': 'ai_handbook.pdf', 'category': 'AI', 'date': '2026-08-15', 'document_type': 'technical'}


In [10]:
# Day 17 - Step 8
# Testing different metadata filters

def show_results(title, query, filters):
    print("\n" + "=" * 60)
    print(title)
    print("=" * 60)

    print("Query:", query)
    print("Filters:", filters)

    results = filtered_retrieve(
        query,
        filters,
        top_k=3
    )

    if not results:
        print("\nNo matching documents found.")
        return

    for i, result in enumerate(results, 1):
        print("\nResult:", i)
        print("Distance:", round(result["distance"], 4))
        print("Source:", result["metadata"]["source"])
        print("Category:", result["metadata"]["category"])
        print("Date:", result["metadata"]["date"])
        print("Document Type:", result["metadata"]["document_type"])
        print("Text:", result["text"][:150])


# ------------------------------------------------
# 1. CATEGORY FILTER
# ------------------------------------------------

show_results(
    "TEST 1 - Category Filter",
    "What is artificial intelligence?",
    {"category": "AI"}
)


# ------------------------------------------------
# 2. SOURCE FILTER
# ------------------------------------------------

show_results(
    "TEST 2 - Source Filter",
    "What is machine learning?",
    {"source": "ai_handbook.pdf"}
)


# ------------------------------------------------
# 3. DOCUMENT TYPE FILTER
# ------------------------------------------------

show_results(
    "TEST 3 - Document Type Filter",
    "What is cloud computing?",
    {"document_type": "technical"}
)


# ------------------------------------------------
# 4. CATEGORY + DATE FILTER
# ------------------------------------------------

show_results(
    "TEST 4 - Category + Date Filter",
    "What is artificial intelligence?",
    {
        "category": "AI",
        "cutoff_date": "2026-01-01"
    }
)


TEST 1 - Category Filter
Query: What is artificial intelligence?
Filters: {'category': 'AI'}

Result: 1
Distance: 0.8199
Source: ai_handbook.pdf
Category: AI
Date: 2026-08-15
Document Type: technical
Text: Machine learning is a branch of artificial intelligence.
        It allows compu

Result: 2
Distance: 1.0432
Source: deep_learning.pdf
Category: AI
Date: 2025-06-20
Document Type: technical
Text: artificial intelligence applications.

Result: 3
Distance: 1.0669
Source: ai_handbook.pdf
Category: AI
Date: 2026-08-15
Document Type: technical
Text:      It allows computers to learn patterns from data and make predictions
      

TEST 2 - Source Filter
Query: What is machine learning?
Filters: {'source': 'ai_handbook.pdf'}

Result: 1
Distance: 0.4851
Source: ai_handbook.pdf
Category: AI
Date: 2026-08-15
Document Type: technical
Text: Machine learning is a branch of artificial intelligence.
        It allows compu

Result: 2
Distance: 0.7962
Source: ai_handbook.pdf
Category: AI
Date: 2026

In [11]:
# Day 17 - Step 9
# Compare Normal Retrieval vs Metadata Filtered Retrieval

test_queries = [
    {
        "query": "What is machine learning?",
        "filters": {"category": "AI"}
    },
    {
        "query": "How is Python used?",
        "filters": {"category": "programming"}
    },
    {
        "query": "What is deep learning?",
        "filters": {"category": "AI"}
    },
    {
        "query": "What is cloud computing?",
        "filters": {"category": "cloud"}
    },
    {
        "query": "How does AI help marketing?",
        "filters": {"category": "marketing"}
    }
]


for i, item in enumerate(test_queries, 1):

    query = item["query"]
    filters = item["filters"]

    print("\n" + "=" * 70)
    print(f"QUERY {i}: {query}")
    print("=" * 70)

    # -----------------------------
    # NORMAL RETRIEVAL
    # -----------------------------

    normal_results = retrieve(
        query,
        top_k=3
    )

    print("\n🔵 WITHOUT METADATA FILTER")

    for j, result in enumerate(normal_results, 1):
        print(
            f"{j}. "
            f"{result['metadata']['source']} | "
            f"{result['metadata']['category']} | "
            f"Distance: {result['distance']:.4f}"
        )

    # -----------------------------
    # FILTERED RETRIEVAL
    # -----------------------------

    filtered_results = filtered_retrieve(
        query,
        filters,
        top_k=3
    )

    print("\n🟢 WITH METADATA FILTER")
    print("Filter:", filters)

    for j, result in enumerate(filtered_results, 1):
        print(
            f"{j}. "
            f"{result['metadata']['source']} | "
            f"{result['metadata']['category']} | "
            f"Distance: {result['distance']:.4f}"
        )


QUERY 1: What is machine learning?

🔵 WITHOUT METADATA FILTER
1. ai_handbook.pdf | AI | Distance: 0.4851
2. ai_handbook.pdf | AI | Distance: 0.7962
3. deep_learning.pdf | AI | Distance: 1.0654

🟢 WITH METADATA FILTER
Filter: {'category': 'AI'}
1. ai_handbook.pdf | AI | Distance: 0.4851
2. ai_handbook.pdf | AI | Distance: 0.7962
3. deep_learning.pdf | AI | Distance: 1.0654

QUERY 2: How is Python used?

🔵 WITHOUT METADATA FILTER
1. python_guide.pdf | programming | Distance: 0.5754
2. python_guide.pdf | programming | Distance: 0.9763
3. ai_handbook.pdf | AI | Distance: 1.1050

🟢 WITH METADATA FILTER
Filter: {'category': 'programming'}
1. python_guide.pdf | programming | Distance: 0.5754
2. python_guide.pdf | programming | Distance: 0.9763
3. python_guide.pdf | programming | Distance: 1.3590

QUERY 3: What is deep learning?

🔵 WITHOUT METADATA FILTER
1. deep_learning.pdf | AI | Distance: 0.6150
2. ai_handbook.pdf | AI | Distance: 0.9205
3. ai_handbook.pdf | AI | Distance: 1.0117

🟢 WITH 

In [12]:
# Day 17 - Step 10
# Create comparison table

import pandas as pd

comparison_data = []

for i, item in enumerate(test_queries, 1):

    query = item["query"]
    filters = item["filters"]

    # Normal retrieval
    normal_results = retrieve(query, top_k=3)

    # Filtered retrieval
    filtered_results = filtered_retrieve(
        query,
        filters,
        top_k=3
    )

    # Categories retrieved without filter
    normal_categories = [
        r["metadata"]["category"]
        for r in normal_results
    ]

    # Categories retrieved with filter
    filtered_categories = [
        r["metadata"]["category"]
        for r in filtered_results
    ]

    comparison_data.append({
        "Query": query,
        "Filter": str(filters),
        "Without Filter": ", ".join(normal_categories),
        "With Filter": ", ".join(filtered_categories),
        "Filtered Results": len(filtered_results)
    })


comparison_df = pd.DataFrame(comparison_data)

comparison_df

,Query,Filter,Without Filter,With Filter,Filtered Results
0,What is machine learning?,{'category': 'AI'},"AI, AI, AI","AI, AI, AI",3
1,How is Python used?,{'category': 'programming'},"programming, programming, AI","programming, programming, programming",3
2,What is deep learning?,{'category': 'AI'},"AI, AI, AI","AI, AI, AI",3
3,What is cloud computing?,{'category': 'cloud'},"cloud, cloud, AI","cloud, cloud, cloud",3
4,How does AI help marketing?,{'category': 'marketing'},"marketing, marketing, marketing","marketing, marketing, marketing",3


In [13]:
# Day 17 - Step 11
# Retrieval Precision Evaluation

# Expected category for each query
expected_categories = [
    "AI",
    "programming",
    "AI",
    "cloud",
    "marketing"
]

evaluation_data = []

for i, item in enumerate(test_queries):

    query = item["query"]
    filters = item["filters"]
    expected_category = expected_categories[i]

    # -----------------------------
    # WITHOUT FILTER
    # -----------------------------

    normal_results = retrieve(query, top_k=3)

    normal_relevant = sum(
        1
        for result in normal_results
        if result["metadata"]["category"] == expected_category
    )

    normal_precision = normal_relevant / len(normal_results)


    # -----------------------------
    # WITH FILTER
    # -----------------------------

    filtered_results = filtered_retrieve(
        query,
        filters,
        top_k=3
    )

    if len(filtered_results) > 0:
        filtered_relevant = sum(
            1
            for result in filtered_results
            if result["metadata"]["category"] == expected_category
        )

        filtered_precision = (
            filtered_relevant / len(filtered_results)
        )
    else:
        filtered_precision = 0


    evaluation_data.append({
        "Query": query,
        "Expected Category": expected_category,
        "Normal Precision": round(normal_precision, 2),
        "Filtered Precision": round(filtered_precision, 2),
        "Normal Precision (%)": f"{normal_precision * 100:.1f}%",
        "Filtered Precision (%)": f"{filtered_precision * 100:.1f}%"
    })


evaluation_df = pd.DataFrame(evaluation_data)

evaluation_df

,Query,Expected Category,Normal Precision,Filtered Precision,Normal Precision (%),Filtered Precision (%)
0,What is machine learning?,AI,1.00,1.0,100.0%,100.0%
1,How is Python used?,programming,0.67,1.0,66.7%,100.0%
2,What is deep learning?,AI,1.00,1.0,100.0%,100.0%
3,What is cloud computing?,cloud,0.67,1.0,66.7%,100.0%
4,How does AI help marketing?,marketing,1.00,1.0,100.0%,100.0%


In [14]:
# Overall precision

normal_avg = evaluation_df["Normal Precision"].mean()
filtered_avg = evaluation_df["Filtered Precision"].mean()

print("Average Normal Precision:")
print(f"{normal_avg * 100:.2f}%")

print("\nAverage Filtered Precision:")
print(f"{filtered_avg * 100:.2f}%")

Average Normal Precision:
86.80%

Average Filtered Precision:
100.00%


In [15]:
# Day 17 - Step 12
# Final Metadata-Aware RAG Demo

def rag_search(query, filters=None, top_k=3):

    results = filtered_retrieve(
        query,
        filters,
        top_k=top_k
    )

    print("\n" + "=" * 70)
    print("METADATA-AWARE RAG SEARCH")
    print("=" * 70)

    print("Query:", query)
    print("Filters:", filters)

    if not results:
        print("\nNo matching documents found.")
        return

    print("\nRetrieved Documents:")

    for i, result in enumerate(results, 1):

        metadata = result["metadata"]

        print("\n" + "-" * 60)
        print(f"Result {i}")

        print("\nText:")
        print(result["text"])

        print("\nMetadata:")
        print("Source:", metadata["source"])
        print("Category:", metadata["category"])
        print("Date:", metadata["date"])
        print("Document Type:", metadata["document_type"])

        print("\nDistance:", round(result["distance"], 4))


# ------------------------------------------------
# DEMO 1
# ------------------------------------------------

rag_search(
    "What is machine learning?",
    {"category": "AI"}
)


# ------------------------------------------------
# DEMO 2
# ------------------------------------------------

rag_search(
    "What is cloud computing?",
    {"category": "cloud"}
)


# ------------------------------------------------
# DEMO 3
# ------------------------------------------------

rag_search(
    "How is Python used?",
    {"category": "programming"}
)


METADATA-AWARE RAG SEARCH
Query: What is machine learning?
Filters: {'category': 'AI'}

Retrieved Documents:

------------------------------------------------------------
Result 1

Text:
Machine learning is a branch of artificial intelligence.
        It allows compu

Metadata:
Source: ai_handbook.pdf
Category: AI
Date: 2026-08-15
Document Type: technical

Distance: 0.4851

------------------------------------------------------------
Result 2

Text:
     It allows computers to learn patterns from data and make predictions
      

Metadata:
Source: ai_handbook.pdf
Category: AI
Date: 2026-08-15
Document Type: technical

Distance: 0.7962

------------------------------------------------------------
Result 3

Text:
Deep learning uses artificial neural networks with multiple layers.
        It i

Metadata:
Source: deep_learning.pdf
Category: AI
Date: 2025-06-20
Document Type: technical

Distance: 1.0654

METADATA-AWARE RAG SEARCH
Query: What is cloud computing?
Filters: {'category': 'cloud